In [94]:
import pandas as pd
import numpy as np
import matplotlib
import rasterio
from tqdm import tqdm
from loguru import logger
import os

In [ ]:
products_df = pd.read_csv("../data/mouherwear-more/products.csv").rename(columns={"id": "product_id"})
products_df.head(2)

,product_id,upc,name,slug,description,is_promotion,is_visible,created_at,updated_at,drophub,attributes
0,1,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN
1,2,6567a397bacd3,شلوار تک پیله فاستونی,Ppnngff,جنس:فاستونی\nرنگبندی:طوسی/مشکی\nسایز۱(دورکمر۶۸...,0,1,2023-11-11 20:45:35,2025-10-23 12:52:37,NaN,NaN


In [ ]:
product_image_df = pd.read_csv("../data/mouherwear-more/product_images.csv").drop(columns=["id"], axis=1)
product_image_df.head(2)

,product_id,path,type,position,created_at,updated_at
0,1,/OKC3DObmdSQWd12MiIkNc9sj8d6Q3k-metaNkY0MUE1MT...,image,3,2024-02-28 14:44:31,2024-04-26 22:59:51
1,1,/dN6KHDq9s2itqX80JEWLbosIYG8Cl7-metaMzA5RkU3OD...,thumbnail,1,2024-02-28 14:46:24,2024-04-26 22:59:51


In [135]:
def sort_group(df):
    return df.sort_values(by="position", ascending=True)

sorted_df = product_image_df.groupby("product_id", group_keys=False).apply(sort_group)
sorted_df.head(2)

/tmp/ipykernel_40924/2268582774.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sorted_df = product_image_df.groupby("product_id", group_keys=False).apply(sort_group)


,product_id,path,type,position,created_at,updated_at
1,1,/dN6KHDq9s2itqX80JEWLbosIYG8Cl7-metaMzA5RkU3OD...,thumbnail,1,2024-02-28 14:46:24,2024-04-26 22:59:51
620,1,/01K06D3VYB2RVJJPTTJP17EJQX.webp,image,1,2025-07-15 10:19:12,2025-07-15 10:19:12


In [136]:
merged_df = sorted_df.merge(products_df, on="product_id", how="left")
merged_df.head(2)

,product_id,path,type,position,created_at_x,updated_at_x,upc,name,slug,description,is_promotion,is_visible,created_at_y,updated_at_y,drophub,attributes
0,1,/dN6KHDq9s2itqX80JEWLbosIYG8Cl7-metaMzA5RkU3OD...,thumbnail,1,2024-02-28 14:46:24,2024-04-26 22:59:51,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN
1,1,/01K06D3VYB2RVJJPTTJP17EJQX.webp,image,1,2025-07-15 10:19:12,2025-07-15 10:19:12,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN


In [116]:
merged_df["path"][0]

'http://cdn.mouherwear1.com/dN6KHDq9s2itqX80JEWLbosIYG8Cl7-metaMzA5RkU3ODMtQzE3My00ODJELUIxNDgtM0QyOUMyOUFFNzRGLmpwZWc=-.jpg'

In [131]:
merged_df["path"] = merged_df["path"].apply(lambda x: "http://cdn.mouherwear1.com" + str(x))
merged_df.head(2)

,product_id,path,type,position,created_at_x,updated_at_x,upc,name,slug,description,is_promotion,is_visible,created_at_y,updated_at_y,drophub,attributes
0,1,http://cdn.mouherwear1.comhttp://cdn.mouherwea...,thumbnail,1,2024-02-28 14:46:24,2024-04-26 22:59:51,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN
1,1,http://cdn.mouherwear1.comhttp://cdn.mouherwea...,image,1,2025-07-15 10:19:12,2025-07-15 10:19:12,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN


In [ ]:
import os
from datetime import datetime
from pathlib import Path
import pandas as pd
import rasterio
from rasterio.errors import RasterioIOError
from tqdm.auto import tqdm   # progress bar


def save_product_images(df, base_dir="../data/images"):
    """
    Save product images AND return a new dataframe with updated local paths.

    Parameters:
        df: pandas.DataFrame with columns:
            - path (URL to image)
            - product_id
            - slug
            - updated_at_y
        base_dir: root folder for saved images

    Returns:
        pandas.DataFrame (with updated 'path' column pointing to saved images)
    """

    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    new_paths = []   # store updated local image paths

    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Saving images"):

        url = row.path
        product_id = str(row.product_id)
        slug = row.slug
        
        # Clean formatted date
        try:
            date_str = pd.to_datetime(row.updated_at_y).strftime("%Y_%m_%d")
        except Exception:
            new_paths.append(None)
            continue

        # Directory for this product
        
        product_dir = base_dir / product_id
        product_dir.mkdir(parents=True, exist_ok=True)

        # Determine next file index
        existing = list(product_dir.glob("*.jpg"))
        file_idx = len(existing) + 1

        filename = f"{file_idx}_{date_str}.jpg"
        save_path = product_dir / filename

        # Update local path in dataframe (even if file already exists)
        new_paths.append(str(save_path))

        # Skip download if already exists
        if save_path.exists():
            continue

        # Load and save image
        try:
            with rasterio.open(url) as src:
                data = src.read()
                if data.shape[0] > 3:
                    continue
                profile = src.profile
        except RasterioIOError:
            continue
        except Exception:
            continue

        # Update profile for JPEG
        profile.update(driver="JPEG", dtype=data.dtype, count=min(data.shape[0], 3))

        try:
            with rasterio.open(save_path, "w", **profile) as dst:
                dst.write(data[:3])  # ensure max 3 bands
        except Exception:
            continue

        # Remove Rasterio sidecar file
        aux = save_path.with_suffix(".jpg.aux.xml")
        if aux.exists():
            aux.unlink()

    # Return updated dataframe
    df = df.copy()
    df["path"] = new_paths

    return df


In [ ]:
df = save_product_images(merged_df)

In [137]:
df.head(2)

,product_id,path,type,position,created_at_x,updated_at_x,upc,name,slug,description,is_promotion,is_visible,created_at_y,updated_at_y,drophub,attributes
0,1,data/images/1/1_2025_10_23.jpg,thumbnail,1,2024-02-28 14:46:24,2024-04-26 22:59:51,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN
1,1,data/images/1/2_2025_10_23.jpg,image,1,2025-07-15 10:19:12,2025-07-15 10:19:12,6567a397ba709,کت,Coat,جنس :فاستونی\nرنگبندی: مشکی/طوسی\nسایز۱(عرض سی...,0,0,2023-11-11 20:28:09,2025-10-23 12:52:09,NaN,NaN


In [ ]:
merged_df.to_csv("merged_product_info.csv")